# Notebook 3 — Leakage-Aware Feature Engineering

## Purpose

Prepare model-ready features for the target `target_default_within_12m` (1 = default within 12 months; 0 = no default). This notebook creates only a clearly defined loan-to-income ratio, retains the existing and previously validated debt-to-income ratio, and uses scikit-learn `Pipeline` and `ColumnTransformer` to impute missing values and one-hot encode categories.

The source data is synthetic. We split rows before fitting preprocessing; the imputer statistics and category vocabulary are learned from training rows only. No predictive model is trained in this notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

relative_data_path = Path("data/raw/finsight_ai_india_synthetic_5000.csv")
data_path = next(
    (path for path in (relative_data_path, Path("..") / relative_data_path) if path.exists()),
    None,
)
if data_path is None:
    raise FileNotFoundError(
        "Could not find data/raw/finsight_ai_india_synthetic_5000.csv. "
        "Run this notebook from the project root or notebooks/ directory."
    )

df = pd.read_csv(data_path)
target_column = "target_default_within_12m"
required_source_columns = {
    target_column,
    "requested_loan_amount_inr",
    "monthly_net_income_inr",
    "debt_to_income_after_loan",
}
missing_required_columns = required_source_columns.difference(df.columns)
if missing_required_columns:
    raise ValueError(f"Required dataset columns are missing: {sorted(missing_required_columns)}")
if df[target_column].isna().any():
    raise ValueError("Training labels are missing; resolve unlabeled rows before creating supervised splits.")
if not df[target_column].isin([0, 1]).all():
    raise ValueError("The target must contain only the binary labels 0 and 1.")
if df["monthly_net_income_inr"].isna().any() or df["monthly_net_income_inr"].le(0).any():
    raise ValueError("Monthly net income must be present and positive to construct an interpretable ratio.")

print(f"Source: {data_path.resolve()}")
print(f"Rows: {len(df):,}; columns: {len(df.columns)}")
print("Target distribution:")
target_summary = df[target_column].value_counts().sort_index().rename(index={0: "No default (0)", 1: "Default (1)"})
display(pd.DataFrame({"applications": target_summary, "percent": (100 * target_summary / len(df)).round(2)}))

print("Missingness in candidate source fields:")
candidate_columns = [
    "employment_years",
    "education_level",
    "credit_history_years",
    "credit_utilization_ratio",
    "savings_balance_inr",
    "banking_relationship_months",
    "repayment_method",
]
candidate_columns = [column for column in candidate_columns if column in df.columns]
missingness = pd.DataFrame(
    {
        "missing_rows": df[candidate_columns].isna().sum(),
        "missing_percent": (100 * df[candidate_columns].isna().mean()).round(2),
    }
).query("missing_rows > 0").sort_values("missing_rows", ascending=False)
display(missingness)

Source: D:\GOURAB DAS\CODE\Git\FinSight AI\data\raw\finsight_ai_india_synthetic_5000.csv
Rows: 5,000; columns: 29
Target distribution:


,applications,percent
target_default_within_12m,,
No default (0),3616,72.32
Default (1),1384,27.68


Missingness in candidate source fields:


,missing_rows,missing_percent
credit_history_years,224,4.48
credit_utilization_ratio,211,4.22
employment_years,184,3.68
savings_balance_inr,152,3.04
education_level,145,2.90
banking_relationship_months,93,1.86
repayment_method,77,1.54


The source contains 5,000 labeled applications. Before choosing predictors, distinguish fields whose meaning is clear from those whose availability at decision time is undocumented. Notebook 1 verified the supplied debt-to-income calculation; it also found that employment duration exceeds age in 10 rows and that some credit, education, savings, banking, and repayment fields are missing. Those records stay in the data here: imputation and category handling happen inside a train-fitted pipeline rather than through row deletion or full-data statistics.

## 1. Define defensible model inputs

Keep the source DTI ratio because Notebook 1 recomputed it as `(existing monthly debt + estimated EMI) / monthly net income`; it matches the supplied field to within rounding precision. Add only **requested loan amount / (12 * monthly net income)**, a dimensionless measure of loan size relative to one year of income. Monthly income is verified positive, so this ratio is defined for all rows.

Use the supplied borrower, application, and loan fields listed below. Treat text categories as nominal: the dataset gives no defensible ordinal category order, so encode them without inventing one. Exclude the target and ID, EAD (an exact copy of requested loan amount), LGD (an unverified loss assumption), prior-default flag (observation window is undocumented), and repayment method (may be observed after application). Confirm feature availability at decision time before deploying any model.

In [2]:
numeric_features = [
    "age_years",
    "employment_years",
    "monthly_net_income_inr",
    "existing_monthly_debt_inr",
    "requested_loan_amount_inr",
    "loan_tenure_months",
    "annual_interest_rate_pct",
    "estimated_emi_inr",
    "debt_to_income_after_loan",
    "credit_history_years",
    "open_credit_accounts",
    "credit_inquiries_last_6m",
    "late_payments_last_12m",
    "credit_utilization_ratio",
    "savings_balance_inr",
    "banking_relationship_months",
    "dependents_count",
]
categorical_features = [
    "state",
    "city_tier",
    "employment_type",
    "education_level",
    "residence_type",
    "loan_purpose",
]
excluded_fields = {
    "application_id": "record identifier; not a borrower characteristic",
    target_column: "outcome label; never an input",
    "prior_default_flag": "outcome observation window and availability are undocumented",
    "repayment_method": "may be determined after application; timing is undocumented",
    "ead_assumption_inr": "duplicates requested_loan_amount_inr exactly",
    "lgd_assumption": "loss assumption; not a verified application-time borrower field",
}
feature_columns = numeric_features + categorical_features

missing_model_columns = set(feature_columns).difference(df.columns)
if missing_model_columns:
    raise ValueError(f"Expected predictor columns are missing: {sorted(missing_model_columns)}")
unexpected_overlap = set(feature_columns).intersection(excluded_fields)
if unexpected_overlap:
    raise ValueError(f"Excluded or target fields entered the feature list: {sorted(unexpected_overlap)}")
if df["monthly_net_income_inr"].le(0).any():
    raise ValueError("The income denominator must be positive for every row.")

print("Feature groups:")
print(f"Numeric source features: {len(numeric_features)}")
print(f"Nominal categorical source features: {len(categorical_features)}")
print("\nExcluded fields and reasons:")
display(pd.Series(excluded_fields, name="decision"))
print("\nExisting numeric binary representations (not new encodings):")
print("prior_default_flag is already numeric, but is deliberately excluded until its observation window is verified.")
print("The target is numeric 0/1 and is kept separately as y, never included in X.")

# Partition the raw, aligned rows before constructing and fitting any learned transformations.
X_raw = df[feature_columns].copy()
y = df[target_column].astype("int8").copy()
X_development_raw, X_test_raw, y_development, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y,
)
X_train_raw, X_validation_raw, y_train, y_validation = train_test_split(
    X_development_raw,
    y_development,
    test_size=(0.15 / 0.85),
    random_state=42,
    stratify=y_development,
)

split_summary = pd.DataFrame(
    {
        "applications": [len(y_train), len(y_validation), len(y_test)],
        "default_rate_percent": [100 * y_train.mean(), 100 * y_validation.mean(), 100 * y_test.mean()],
    },
    index=["Training", "Validation", "Test"],
)
print("Stratified splits (70% training, 15% validation, 15% test):")
display(split_summary.round(2))
assert len(X_train_raw) + len(X_validation_raw) + len(X_test_raw) == len(df)
assert X_train_raw.index.intersection(X_validation_raw.index).empty
assert X_train_raw.index.intersection(X_test_raw.index).empty
assert X_validation_raw.index.intersection(X_test_raw.index).empty

Feature groups:
Numeric source features: 17
Nominal categorical source features: 6

Excluded fields and reasons:


application_id                record identifier; not a borrower characteristic
target_default_within_12m                        outcome label; never an input
prior_default_flag           outcome observation window and availability ar...
repayment_method             may be determined after application; timing is...
ead_assumption_inr                duplicates requested_loan_amount_inr exactly
lgd_assumption               loss assumption; not a verified application-ti...
Name: decision, dtype: str


Existing numeric binary representations (not new encodings):
prior_default_flag is already numeric, but is deliberately excluded until its observation window is verified.
The target is numeric 0/1 and is kept separately as y, never included in X.
Stratified splits (70% training, 15% validation, 15% test):


,applications,default_rate_percent
Training,3500,27.66
Validation,750,27.73
Test,750,27.73


The predictor list uses 17 numeric and six nominal source fields. `debt_to_income_after_loan` is retained as a supplied, validated ratio; the only additional feature will compare the requested principal with one year of net income. Target, ID, duplicate exposure assumption, unverified loss assumption, and fields with uncertain outcome/decision-time availability are excluded. Stratified 70/15/15 partitions are created before learning any preprocessing parameters; the validation and test sets remain untouched until transformation.

In [3]:
engineered_numeric_features = ["requested_loan_to_annual_income"]

def add_loan_to_annual_income(frame):
    """Return a copy with requested principal divided by annualized net income."""
    result = frame.copy()
    monthly_income = result["monthly_net_income_inr"]
    invalid_income = monthly_income.isna() | monthly_income.le(0)
    if invalid_income.any():
        raise ValueError(
            "Cannot calculate requested_loan_to_annual_income: monthly net income "
            "must be present and positive."
        )
    result["requested_loan_to_annual_income"] = (
        result["requested_loan_amount_inr"] / (12 * monthly_income)
    )
    if not np.isfinite(result["requested_loan_to_annual_income"]).all():
        raise ValueError("The engineered loan-to-income ratio contains a non-finite value.")
    return result

# This deterministic calculation has no learned parameters; apply it to each split after partitioning.
X_train = add_loan_to_annual_income(X_train_raw)
X_validation = add_loan_to_annual_income(X_validation_raw)
X_test = add_loan_to_annual_income(X_test_raw)

all_numeric_features = numeric_features + engineered_numeric_features
print("Engineered feature formula:")
print("requested_loan_to_annual_income = requested_loan_amount_inr / (12 * monthly_net_income_inr)")
ratio_summary = X_train["requested_loan_to_annual_income"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.95]
).to_frame("training applications")
display(ratio_summary.round(4))
print("\nNon-missing engineered ratios per split:")
display(
    pd.DataFrame(
        {
            "training": X_train["requested_loan_to_annual_income"].notna().sum(),
            "validation": X_validation["requested_loan_to_annual_income"].notna().sum(),
            "test": X_test["requested_loan_to_annual_income"].notna().sum(),
        },
        index=["requested_loan_to_annual_income"],
    )
)

Engineered feature formula:
requested_loan_to_annual_income = requested_loan_amount_inr / (12 * monthly_net_income_inr)


,training applications
count,3500.0000
mean,0.7905
std,1.0370
min,0.0157
25%,0.2472
50%,0.4596
75%,0.9299
95%,2.5073
max,15.8811



Non-missing engineered ratios per split:


,training,validation,test
requested_loan_to_annual_income,3500,750,750


`requested_loan_to_annual_income` has a direct interpretation: the requested principal as a fraction of annualized net income. It is calculated separately within each already-separated partition and uses no fitted statistic; the data-dependent imputers and encoders are still learned later from training rows alone. The verified source DTI ratio remains available because it measures **monthly obligations relative to monthly income**, a related but distinct concept.

## 2. Build train-fitted preprocessing pipelines

Use a numeric pipeline with median imputation, missingness indicators, and standardization. Use a categorical pipeline that assigns missing categories an explicit value and one-hot encodes categories unseen during training safely. `ColumnTransformer` applies each pipeline to its declared fields. Only `X_train` is supplied to `fit_transform`; the validation and test partitions are passed to `transform` after training-only statistics and categories have been learned.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, all_numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)

# This is the only fit call: validation and test data are transform-only.
X_train_transformed = preprocessor.fit_transform(X_train)
X_validation_transformed = preprocessor.transform(X_validation)
X_test_transformed = preprocessor.transform(X_test)

train_feature_names = preprocessor.get_feature_names_out()
print(f"Input numeric fields, including the engineered ratio: {len(all_numeric_features)}")
print(f"Input nominal categorical fields: {len(categorical_features)}")
print(f"Output model-ready columns after missing indicators and one-hot encoding: {len(train_feature_names)}")
print("Transformed matrix shapes (rows, features):")
display(
    pd.DataFrame(
        {
            "rows": [X_train_transformed.shape[0], X_validation_transformed.shape[0], X_test_transformed.shape[0]],
            "features": [X_train_transformed.shape[1], X_validation_transformed.shape[1], X_test_transformed.shape[1]],
        },
        index=["Training", "Validation", "Test"],
    )
)
print("\nExample output feature names:")
print("\n".join(train_feature_names[:20]))
print("\nNumeric training medians learned by the pipeline:")
numeric_imputer = preprocessor.named_transformers_["numeric"].named_steps["imputer"]
display(pd.Series(numeric_imputer.statistics_, index=all_numeric_features, name="training median").to_frame().round(4))

Input numeric fields, including the engineered ratio: 18
Input nominal categorical fields: 6
Output model-ready columns after missing indicators and one-hot encoding: 68
Transformed matrix shapes (rows, features):


,rows,features
Training,3500,68
Validation,750,68
Test,750,68



Example output feature names:
numeric__age_years
numeric__employment_years
numeric__monthly_net_income_inr
numeric__existing_monthly_debt_inr
numeric__requested_loan_amount_inr
numeric__loan_tenure_months
numeric__annual_interest_rate_pct
numeric__estimated_emi_inr
numeric__debt_to_income_after_loan
numeric__credit_history_years
numeric__open_credit_accounts
numeric__credit_inquiries_last_6m
numeric__late_payments_last_12m
numeric__credit_utilization_ratio
numeric__savings_balance_inr
numeric__banking_relationship_months
numeric__dependents_count
numeric__requested_loan_to_annual_income
numeric__missingindicator_employment_years
numeric__missingindicator_credit_history_years

Numeric training medians learned by the pipeline:


,training median
age_years,43.0000
employment_years,5.6000
monthly_net_income_inr,39576.5000
existing_monthly_debt_inr,11271.5000
requested_loan_amount_inr,217453.0000
loan_tenure_months,36.0000
annual_interest_rate_pct,16.6000
estimated_emi_inr,9679.5000
debt_to_income_after_loan,0.6340
credit_history_years,3.3000


All three partitions have the same encoded feature width, because the encoder's vocabulary is fixed by the training partition and unknown validation/test categories are ignored safely. Numeric missingness is imputed with training medians and accompanied by indicators; missing categories receive a dedicated level. Scaling, imputation, and category discovery all live inside a single reusable `ColumnTransformer`, making the learned transformation available for a later model pipeline without leaking holdout statistics.

## 3. Validate leakage controls and engineered features

Check partition isolation, target exclusion, feature-name clarity, finite transformed values, and missing-data behavior. Finally, send a deliberately unseen state through the fitted transformer to confirm one-hot encoding does not fail on categories absent from training.

In [5]:
for split_name, features, labels, transformed in [
    ("training", X_train, y_train, X_train_transformed),
    ("validation", X_validation, y_validation, X_validation_transformed),
    ("test", X_test, y_test, X_test_transformed),
]:
    assert target_column not in features.columns
    assert not set(excluded_fields).intersection(features.columns)
    assert len(features) == len(labels) == transformed.shape[0]
    assert np.isfinite(transformed.data if hasattr(transformed, "data") and hasattr(transformed, "tocsr") else transformed).all()

assert X_train.index.intersection(X_validation.index).empty
assert X_train.index.intersection(X_test.index).empty
assert X_validation.index.intersection(X_test.index).empty
assert preprocessor.n_features_in_ == X_train.shape[1]
assert X_train_transformed.shape[1] == X_validation_transformed.shape[1] == X_test_transformed.shape[1]
assert not any(target_column in feature_name for feature_name in train_feature_names)

# A category absent from training must be ignored, not mapped into a false known category or rejected.
unseen_category_probe = X_test.iloc[[0]].copy()
unseen_category_probe.loc[:, "state"] = "Unseen test-only state"
unseen_category_transformed = preprocessor.transform(unseen_category_probe)
assert unseen_category_transformed.shape[1] == X_train_transformed.shape[1]
assert np.isfinite(
    unseen_category_transformed.data
    if hasattr(unseen_category_transformed, "tocsr")
    else unseen_category_transformed
).all()

missing_indicator_names = [name for name in train_feature_names if "missingindicator" in name.lower()]
encoded_category_names = [name for name in train_feature_names if "categorical__" in name]
validation_report = pd.DataFrame(
    {
        "check": [
            "Target absent from model input",
            "Identifier and unverified assumptions absent",
            "Train / validation / test rows are disjoint",
            "Holdout matrices match training feature width",
            "Output has no NaN or infinite values",
            "Unseen category transforms successfully",
            "Numeric missingness indicators created",
            "One-hot categorical columns created",
        ],
        "result": [
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            f"{len(missing_indicator_names)} indicators",
            f"{len(encoded_category_names)} one-hot columns",
        ],
    }
)
display(validation_report)
print("All leakage, shape, finite-value, and unseen-category assertions passed.")

,check,result
0,Target absent from model input,PASS
1,Identifier and unverified assumptions absent,PASS
2,Train / validation / test rows are disjoint,PASS
3,Holdout matrices match training feature width,PASS
4,Output has no NaN or infinite values,PASS
5,Unseen category transforms successfully,PASS
6,Numeric missingness indicators created,5 indicators
7,One-hot categorical columns created,45 one-hot columns


All leakage, shape, finite-value, and unseen-category assertions passed.


The checks confirm that the target and excluded columns are not model inputs, the three data partitions do not overlap, and the same training-fitted transformation safely handles every holdout row—including a category never seen in training. Imputed numeric fields preserve their missingness signal. These checks validate the preprocessing design; they do not measure predictive performance or replace confirmation that each feature is available at application time.

## Feature Engineering Summary

### Q&A

- **Which derived feature was added?** `requested_loan_to_annual_income = requested_loan_amount_inr / (12 * monthly_net_income_inr)`, a dimensionless ratio of requested principal to annualized net income. Positive income is checked before the ratio is calculated.
- **How are missing and categorical inputs handled?** A scikit-learn `ColumnTransformer` applies a numeric median imputer with missingness indicators and scaling, plus a categorical constant imputer and one-hot encoder configured to ignore unseen levels.
- **How is transformation leakage prevented?** Stratified train/validation/test splits are made before fitting; only training rows fit the imputer, scaler, and category vocabulary. Validation and test partitions are transform-only.

### Data Analysis Key Findings

- The validated source `debt_to_income_after_loan` is retained rather than duplicated: it represents monthly debt plus EMI divided by monthly net income. The new loan-to-annual-income ratio measures a distinct principal-versus-annual-income quantity.
- The target, application ID, duplicate EAD assumption, unverified LGD assumption, prior-default flag with undocumented timing, and repayment method with undocumented timing are excluded from predictors.
- The train-fitted transformations output matching, finite feature matrices for all splits, add numeric missingness indicators, preserve missing categories, and successfully handle an unseen validation/test category.

### Insights or Next Steps

- Reuse `preprocessor` inside the eventual estimator pipeline and evaluate only on the reserved validation/test partitions; do not refit the transformer on holdout data.
- Before interpreting or deploying a model, verify at application time the availability and business definition of every included feature, especially loan pricing and post-loan fields. This synthetic exercise is not evidence about real applicants.